# Oppitunti 10 - AI-agentit tuotannossa

Tässä oppitunnissa opit **tuotantomalleja** AI-agenteille Microsoft Agent Frameworkin (Python) avulla. Käsittelemme:

- **Havainnointi** — ajoituksen ja lokituksen lisääminen agenttien vuorovaikutuksiin
- **Arviointi** — arvioiva agentti vastausten laadun pisteyttämiseen
- **Kustannusten hallinta** — strategiat tokenien optimointiin ja mallin valintaan

Tapaus on **matkatoimisto**, joka auttaa käyttäjiä suunnittelemaan matkoja, valvonnan ja arvioinnin kerrostuessa päälle.


## Asennus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Tuotantoon siirtymisen näkökohdat

Siirryttäessä tekoälyagenttien prototyypeistä tuotantoon on kiinnitettävä huomiota kolmeen pilariseen:

1. **Havaittavuus** — Tarvitset näkyvyyden siihen, mitä agentti tekee, kuinka kauan se vie aikaa ja mitä työkaluja se kutsuu. Ilman jäljitystä ja lokitusta tuotanto-ongelmien virheenkorjaus on lähes mahdotonta.

2. **Evaluointi** — Automaattiset laadunvalvontatarkistukset varmistavat, että agentin vastaukset pysyvät ajan myötä tarkkoina, täydellisinä ja hyödyllisinä. Arvioija-agentti voi pisteyttää vastaukset määriteltyjen kriteerien mukaisesti.

3. **Kustannusten hallinta** — Tokenien käyttö vaikuttaa suoraan kustannuksiin. Strategiat kuten kehotteen optimointi, mallin valinta ja välimuisti auttavat pitämään kulut hallinnassa ilman laadun heikentämistä.


## Havainnoitavan agentin rakentaminen

Määrittelemme matkustustyökalut ja ympäröimme agenttikutsun ajoituksella, jotta voimme seurata viivettä. Tuotannossa integroisit OpenTelemetryn tai vastaavan jäljitystaustan kanssa.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Arviointimallit

Yleinen tuotantokuvio on käyttää toista agenttia **arvioijana**. Arvioija pisteyttää ensisijaisen agentin vastauksen ennakkoon määriteltyjen kriteerien, kuten täydellisyyden, tarkkuuden ja hyödyllisyyden, perusteella.

Tämä mahdollistaa:
- Automaattiset laadunportit ennen kuin vastaukset saavuttavat käyttäjät
- Takaisinkytkentähäiriön havaitsemisen, kun kehotteet tai mallit muuttuvat
- Agentin suorituskyvyn jatkuvan seurannan ajan kuluessa


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kustannusten hallintastrategiat

Kustannusten hallinta on ratkaisevan tärkeää tuotannon tekoäly-agenttien kohdalla. Tässä ovat keskeiset strategiat:

| Strategia | Kuvaus |
|---|---|
| **Kehoteoptimointi** | Pidä järjestelmäohjeet tiiviinä. Poista tarpeeton konteksti vähentääksesi syötteen tokeneita. |
| **Mallin valinta** | Käytä pienempiä, halvempia malleja (esim. GPT-4.1-mini) yksinkertaisiin tehtäviin, kuten luokitteluun tai tiedon poimintaan, ja varaa suuremmat mallit monimutkaiseen päättelyyn. |
| **Välimuisti** | Välimuistita työkalujen tulokset ja usein kysytyt haut välttääksesi päällekkäisiä API-kutsuja. |
| **Token-budjetit** | Aseta `max_tokens`-rajoitukset estämään odottamattoman pitkiä vastauksia. |
| **Ryvästäminen** | Ryhmittele useita käyttäjäkyselyitä yhteen API-kutsuun, kun se on mahdollista. |

Käytännössä kerroksittainen lähestymistapa toimii hyvin: ohjaa yksinkertaiset pyynnöt nopealle, edulliselle mallille ja vain monimutkaiset kyselyt etenevät kyvykkäämmälle mallille.


## Yhteenveto

Tässä oppitunnissa opit, kuinka:

1. **Lisätään havaittavuutta** agenttien vuorovaikutuksiin ajoituksella ja lokituksella, luoden pohjan jäljitykselle ja valvonnalle.
2. **Arvioidaan agenttien vastauksia** automaattisesti käyttämällä arvioija-agenttia, joka pisteyttää täydellisyyden, tarkkuuden ja hyödyllisyyden.
3. **Hallitset kustannuksia** kehoteoptimoinnin, mallivalinnan, välimuistin ja token-budjettien avulla.

Nämä tuotantomallit auttavat varmistamaan, että tekoälyagenttisi ovat luotettavia, mitattavia ja kustannustehokkaita mittakaavassa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
